In [13]:
import numpy as np
import pandas as pd

# ----------------------------
# INPUTS
# ----------------------------
# Truck factors provided by user (unit: kg CO2 / (t·km)) for years 2016-2025 from 
emission_factors_truck_input = pd.DataFrame([
    {"year": 2016, "truck_factor": 0.08450},
    {"year": 2017, "truck_factor": 0.08412 },
    {"year": 2018, "truck_factor": 0.08231 },
    {"year": 2019, "truck_factor": 0.08020},
    {"year": 2020, "truck_factor": 0.07938 },
    {"year": 2021, "truck_factor": 0.07895 },
    {"year": 2022, "truck_factor": 0.07890 },
    {"year": 2023, "truck_factor": 0.07294 },
    {"year": 2024, "truck_factor": 0.07321},
    {"year": 2025, "truck_factor": 0.07576},
])

# Observed aircraft fuel intensity (g fuel / (t·km)) 
# from table FUEL BURN OF NEW COMMERCIAL JET AIRCRAFT: 1960 TO 2019 ICCT
air_observed = pd.DataFrame([
    {"year": 2000, "fuel_g_tkm": 72},
    {"year": 2001, "fuel_g_tkm": 72},
    {"year": 2002, "fuel_g_tkm": 71},
    {"year": 2003, "fuel_g_tkm": 75},
    {"year": 2004, "fuel_g_tkm": 74},
    {"year": 2005, "fuel_g_tkm": 73},
    {"year": 2006, "fuel_g_tkm": 72},
    {"year": 2007, "fuel_g_tkm": 71},
    {"year": 2008, "fuel_g_tkm": 70},
    {"year": 2009, "fuel_g_tkm": 69},
    {"year": 2010, "fuel_g_tkm": 69},
    {"year": 2011, "fuel_g_tkm": 68},
    {"year": 2012, "fuel_g_tkm": 66},
    {"year": 2013, "fuel_g_tkm": 66},
    {"year": 2014, "fuel_g_tkm": 66},
    {"year": 2015, "fuel_g_tkm": 66},
    {"year": 2016, "fuel_g_tkm": 66},
    {"year": 2017, "fuel_g_tkm": 64},
    {"year": 2018, "fuel_g_tkm": 62},
    {"year": 2019, "fuel_g_tkm": 60},
    {"year": 2020, "fuel_g_tkm": 59},
])

# Constants
ICAO_KGCO2_PER_KG_FUEL = 3.16  # standard conversion from jet fuel to CO2, ICAO Methodology

# ----------------------------
# Fit models
# ----------------------------
years_full = np.arange(2000, 2025)

# --- Air: linear on grams ---
coef_air_lin = np.polyfit(air_observed['year'].values, air_observed['fuel_g_tkm'].values, 1)
air_fuel_pred_lin = np.polyval(coef_air_lin, years_full)

# --- Air: log-linear on grams (fit ln(fuel_g_tkm)) ---
coef_air_log = np.polyfit(air_observed['year'].values, np.log(air_observed['fuel_g_tkm'].values), 1)
air_fuel_pred_log = np.exp(np.polyval(coef_air_log, years_full))

# Convert fuel (g/(t·km)) -> CO2 kg/(t·km)
air_co2_pred_lin = (air_fuel_pred_lin / 1000.0) * ICAO_KGCO2_PER_KG_FUEL
air_co2_pred_log = (air_fuel_pred_log / 1000.0) * ICAO_KGCO2_PER_KG_FUEL

truck_obs = emission_factors_truck_input[['year','truck_factor']].copy()
# log-linear fit for truck (fit log of truck factor)
coef_truck_log = np.polyfit(truck_obs['year'].values, np.log(truck_obs['truck_factor'].values), 1)
truck_pred_log = np.exp(np.polyval(coef_truck_log, years_full))

# ----------------------------
# Build unified DataFrame
# ----------------------------
df_out = pd.DataFrame({
    'year': years_full,
    # Air (fuel grams)
    'fuel_g_tkm observation': [air_observed.set_index('year').loc[y]['fuel_g_tkm'] if y in air_observed['year'].values else np.nan for y in years_full],
    'fuel_g/tkm_ estimated': air_fuel_pred_log,
    # Air CO2 kg/(t·km)
    'air_factor (CO2_kg/tkm)': air_co2_pred_log,
    # Truck factors
    'truck_factor (CO2_kg/tkm)': truck_pred_log
})

# Clip unrealistic negatives
for c in df_out.columns:
    if c != 'year':
        df_out[c] = pd.to_numeric(df_out[c], errors='coerce')

# Save CSV
csv_path = 'emission_factors_2000_2025.csv'
df_out.to_csv(csv_path, index=False)

# Done
print('CSV saved to:', csv_path)


CSV saved to: emission_factors_2000_2025.csv


## How to use this data

The file `emission_factors_2000_2025.csv` provides modeled emission intensities for both **air cargo** and **road freight (truck)** used in Formula 1 logistics analysis.  
These values represent the average amount of **CO₂ emitted per tonne of cargo transported over one kilometre**:

$$
\text{Emission Factor} = \text{kg CO₂ / (t} \times \text{ km)}
$$

---

### Parameters and Units

| Symbol | Description | Units |
|:--|:--|:--|
| \(E_{CO₂}\) | Total CO₂ emissions | kg CO₂ |
| \(f\) | Emission factor (from dataset) | kg CO₂ / (t·km) |
| \(M\) | Cargo mass | t (tonnes) |
| \(D\) | Distance travelled | km |

---

### Core Equation

To estimate total emissions for a transport leg:

$$

E_{CO₂} = f \times M \times D

$$

where:
- **f** = emission factor (air or truck, from the dataset)
- **M** = cargo mass in tonnes
- **D** = distance in km between two circuits

---

**Example: Air Freight**

A Formula 1 team ships **50 tonnes** of equipment over **8,000 km**.  
From the dataset, the 2023 air factor is **0.48 kg CO₂ / (t·km)**.

$$ 
E_{CO₂} = 0.48 \times 50 \times 8{,}000 = 192{,}000\ \text{kg CO₂} = 192\ \text{t CO₂}
$$

---

**Example: Truck Freight (European races)**

For **50 t** of cargo moved **1,200 km** by truck in **2023**, with a factor of **0.073 kg CO₂ / (t·km)**:

$$
E_{CO₂} = 0.073 \times 50 \times 1{,}200 = 4{,}380\ \text{kg CO₂} = 4.38\ \text{t CO₂}
$$

---
